In [1]:
print("Tool Calling")

Tool Calling


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")

In [3]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash'
)

d:\Playground\Agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# define tools
from llama_index.core.tools import FunctionTool

def add(x: int, y:int) -> int:
    """
    Args: x, y integers
    Returns: Addition Int
    """
    return x+y

def mystery(x:int, y:int) -> int:
    """
    Args: x, y integers
    Returns: value of operation in int    
    """
    return (x-y)/(x+y)

add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [5]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash'
)

response = llm.predict_and_call(
    [add_tool, mystery_tool],
    "Tell me output of mystery function on 2 and 9", verbose=True
)

print(response)

=== Calling Function ===
Calling function: mystery with args: {"y": 9.0, "x": 2.0}
=== Function Output ===
-0.6363636363636364
-0.6363636363636364


In [6]:
# Load Data
from llama_index.core import SimpleDirectoryReader
document = SimpleDirectoryReader(input_files=['../Data/CAG.pdf']).load_data()

In [7]:
# Split data
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes  =splitter.get_nodes_from_documents(document)

In [8]:
print(nodes[0].get_content(metadata_mode='all'))

page_label: 1
file_name: CAG.pdf
file_path: ..\Data\CAG.pdf
file_type: application/pdf
file_size: 116336
creation_date: 2025-03-13
last_modified_date: 2025-01-19

arXiv:2412.15605v1  [cs.CL]  20 Dec 2024
Don’t Do RAG:
When Cache-Augmented Generation is All You Need for
Knowledge Tasks
Brian J Chan ∗
Chao-Ting Chen∗
Jui-Hung Cheng ∗
Department of Computer Science
National Chengchi University
Taipei, Taiwan
{110703065,110703038,110703007}@nccu.edu.tw
Hen-Hsen Huang
Insititue of Information Science
Academia Sinica
Taipei, Taiwan
hhhuang@iis.sinica.edu.tw
Abstract
Retrieval-augmented generation (RAG) has gained tractionas a
powerful approach for enhancing language models by integra ting
external knowledge sources. However, RAG introduces chall enges
such as retrieval latency, potential errors in document sel ection,
and increased system complexity. With the advent of large la n-
guage models (LLMs) featuring signiﬁcantly extended conte xt win-
dows, this paper proposes an alternative parad

In [9]:
# embedding model
from llama_index.embeddings.gemini import GeminiEmbedding

embed_model = GeminiEmbedding(model='model/embedding-001')

In [10]:
# Define vector store index
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes, embed_model=embed_model)
query_engine = vector_index.as_query_engine(similarity_top_k=2, llm=llm)

In [11]:
# Metadata filtering
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    llm=llm,
    filters=MetadataFilters.from_dicts(
        [
            {'key':'page_label', 'value': '2'}
        ]
    )
)


response = query_engine.query(
    "explain methodology in CAG"
)

print(response)

The CAG framework uses the extended context capabilities of long-context LLMs to enable retrieval-free knowledge integration. It addresses the computational challenges and inefficiencies inherent in traditional RAG systems by preloading external knowledge sources and precomputing the key-value (KV) cache. The operation of this framework is divided into three phases:

(1) External Knowledge Preloading: A collection of documents relevant to the target application is preprocessed and formatted to fit within the model’s extended context window. The LLM processes the documents, transforming them into a precomputed KV cache, which is stored on disk or in memory for future use. The computational cost of processing is incurred only once, regardless of the number of subsequent queries.

(2) Inference: During inference, the precomputed KV cache is loaded alongside the user’s query. The LLM utilizes this cached context to generate responses. By preloading the external knowledge, this phase elimin

In [22]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '2', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}


In [12]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash',
    temperature=0
)

In [23]:
# Define Auto Retrieval
from typing import List
from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    if page_numbers:
        metadata_dicts = [
            {"key": "page_label", "value": p} for p in page_numbers
        ]
    
        filters = MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )

        query_engine = vector_index.as_query_engine(
            similarity_top_k=2,
            filters=filters,
        )
    else:
        query_engine = vector_index.as_query_engine(
            similarity_top_k=2
        )
    response = query_engine.query(query)
    return response
    

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [21]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-3.5-turbo", temperature=0)
# llm = Gemini(model='models/gemini-2.0-flash', temperature=0)

response = llm.predict_and_call(
    tools=[vector_query_tool],  # Wrap in list and use named parameter
    user_msg="What is the experimental setup in CAG as described on page 3?",
    verbose=True
)

=== Calling Function ===
Calling function: vector_tool with args: {"query": "experimental setup", "page_numbers": ["3"]}
=== Function Output ===
The experiments were conducted using the Stanford Question Answering Dataset (SQuAD) 1.0 and the HotPotQA dataset. These datasets were chosen for their distinct challenges, with SQuAD focusing on precise answers within single passages and HotPotQA emphasizing multi-hop reasoning across multiple documents. The experiments involved creating test sets with varying reference text lengths to assess retrieval difficulty. The experiments were carried out using the Llama 3.1 8B Instruction model for both the baseline RAG systems and the proposed method, with the context of each dataset preloaded into the model via a precomputed key-value cache.


In [22]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '3', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '3', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}


In [32]:
# Add some other tool
from llama_index.core import SummaryIndex # for getting summary
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode = "tree_summarize",
    use_async=True,
)

summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=("useful to get summary of a CAG (cache augmented generation) paper")
)

In [25]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "How was the Baseline system of a CAG?", 
    verbose=True
)

=== Calling Function ===
Calling function: summary_tool with args: {"input": "Baseline system of a CAG"}
=== Function Output ===
The baseline system of a CAG includes two retrieval strategies: BM25 for sparse retrieval and OpenAI Indexes for dense retrieval. The BM25 algorithm ranks documents based on term frequency-inverse document frequency (TF-IDF) and document length normalization, while OpenAI Indexes use dense embeddings to represent documents and queries in a shared semantic space.


In [26]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '1', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '1', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '2', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '2', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '3', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_lab

In [36]:
response = llm.predict_and_call(
    tools=[vector_query_tool, summary_tool], 
    user_msg="Use the summary tool to give me a summary of the paper.", 
    verbose=True
)

=== Calling Function ===
Calling function: summary_tool with args: {"input": "cache augmented generation"}
=== Function Output ===
The cache-augmented generation (CAG) approach involves preloading all relevant resources into the large language model's extended context and caching its runtime parameters. This method eliminates retrieval latency, minimizes retrieval errors, and simplifies system architecture while maintaining high-quality responses by ensuring the model processes all relevant context holistically.


In [31]:
# Direct use of the summary tool
summary_response = summary_tool.call("What is the paper about?")
print(summary_response)

The paper discusses a new approach called Cache-Augmented Generation (CAG) as an alternative to Retrieval-Augmented Generation (RAG) for enhancing language models by integrating external knowledge sources. CAG involves preloading all relevant resources into the model's extended context and caching its runtime parameters, eliminating the need for real-time retrieval during inference. The method aims to address challenges such as retrieval latency, errors in document selection, and increased system complexity associated with traditional RAG systems. The paper highlights that CAG can provide comparable or superior results with reduced complexity, particularly for applications with a constrained knowledge base.
